In [ ]:
# Install CapyMOA (https://capymoa.org/setup/)
%pip install -q capymoa==0.15.1

In [ ]:
%%writefile settings_autoclass.json
{

  "windowSize" :  1000,
  "ensembleSize" : 10,
  "newConfigurations" : 10,
  "keepCurrentModel" : true,
  "lambda" : 0.05,
  "preventAlgorithmDeath" : true,
  "keepGlobalIncumbent" : true,
  "keepAlgorithmIncumbents" : true,
  "keepInitialConfigurations" : true,
  "useTestEnsemble" : true,
  "resetProbability" : 0.01,
  "numberOfCores" : 1,
  "performanceMeasureMaximisation": true,

  "algorithms": [
    {
      "algorithm": "moa.classifiers.lazy.kNN",
      "parameters": [
        {"parameter": "k", "type":"integer", "value":10, "range":[2,30]}
      ]
    }
  ,
    {
      "algorithm": "moa.classifiers.trees.HoeffdingTree",
      "parameters": [
        {"parameter": "g", "type":"integer", "value":200, "range":[10, 200]},
       {"parameter": "c", "type":"float", "value":0.01, "range":[0, 1]}
      ]
    }
  ,
    {
      "algorithm": "moa.classifiers.lazy.kNNwithPAWandADWIN",
      "parameters": [
        {"parameter": "k", "type":"integer", "value":10, "range":[2,30]}
      ]
    }
    ,
    {
      "algorithm": "moa.classifiers.trees.HoeffdingAdaptiveTree",
      "parameters": [
        {"parameter": "g", "type":"integer", "value":200, "range":[10, 200]},
        {"parameter": "c", "type":"float", "value":0.01, "range":[0, 1]}
      ]
    }

  ]
}


# Save and Load a Model

In this tutorial, we illustrate the process of saving and loading a model using CapyMOA. 

* We use the SEA synthetic generator as the data source, and the AdaptiveRandomForestClassifier as the learner.
* The trained model is saved to a file, specifically `capymoa_ARF_model.pkl`.
* Subsequently, we reload the model from the file and resume training and evaluating its performance on the SEA data.
* As a final step, we delete the model file.

---

*More information about CapyMOA can be found at* https://www.capymoa.org.

**last update on 01/12/2025**

## Training and saving the model

* We train the model on 5k instances from SEA using the `evaluate_prequential` function.
* We proceed to save the model with `save_model(learner, "capymoa_ARF_model.pkl")`.

In [ ]:
from capymoa.classifier import AdaptiveRandomForestClassifier
from capymoa.core.io import load_model, save_model
from capymoa.evaluation import prequential_evaluation
from capymoa.stream.generator import SEA

stream = SEA()
learner = AdaptiveRandomForestClassifier(schema=stream.get_schema(), ensemble_size=10)

results = prequential_evaluation(stream=stream, learner=learner, max_instances=5000)

print(f"Accuracy: {results['cumulative'].accuracy():.2f}")

with open("capymoa_ARF_model.pkl", "wb") as f:
    save_model(learner, f)

## Loading and resuming training

* We use `os.path.getsize()` to inspect the size (KB) of the saved file.
* We don't restart the synthetic stream, we just continue processing it through another call to `prequential_evaluation`.
* Finally, we observe the accuracy.

In [ ]:
import os

model_file = "capymoa_ARF_model.pkl"

model_size = os.path.getsize(model_file)
print(f"The saved model size: {model_size / 1024:.2f} KB")

with open(model_file, "rb") as f:
    restored_learner = load_model(f)

# Train for more 50k instances on the restored model
results = prequential_evaluation(
    stream=stream, learner=restored_learner, max_instances=5000
)

print(f"Updated accuracy: {results['cumulative'].accuracy():.2f}")

## Cleanup 

* As a last step, we delete the model.

In [ ]:
if os.path.exists(model_file):
    os.remove(model_file)
    print(f"File {model_file} has been deleted.")
else:
    print(f"File {model_file} not found.")